## 15_env_check — run FIRST in BOTH workspaces (old Free Edition / new Azure Databricks)

Read-only diagnostic: runtime versions, feature probes, table/model naming resolution.
Creates nothing and writes nothing; safe on any compute. Zero cost.
Why: the OLD workspace is Databricks Free Edition (serverless, Python 3.10+,
     Spark 3.5.x, Unity Catalog); the NEW workspace is Azure Databricks
     (Premium + UC), reached through the Azure Bastion jumphost.


In [ ]:


dbutils.widgets.text("workspace_label", "old-free", "Label: old-free | new-azure")
dbutils.widgets.text("SCHEMA", "practice", "Schema used in resolved table names")

WORKSPACE_LABEL = dbutils.widgets.get("workspace_label").strip()
SCHEMA = dbutils.widgets.get("SCHEMA").strip()

print(f"ENV CHECK — workspace: {WORKSPACE_LABEL}")


In [ ]:


import sys
import pandas as pd


def safe(fn, default="(unavailable)"):
    try:
        return fn()
    except Exception as e:
        return f"(unavailable: {str(e)[:80]})"


def conf(key):
    return safe(lambda: spark.conf.get(key))


versions = pd.DataFrame([
    {"item": "workspace label", "value": WORKSPACE_LABEL},
    {"item": "runtime (spark.databricks.runtimeVersion)", "value": conf("spark.databricks.runtimeVersion")},
    {"item": "Spark version tag", "value": conf("spark.databricks.clusterUsageTags.SparkVersion")},
    {"item": "spark.version", "value": spark.version},
    {"item": "Python", "value": sys.version.split()[0]},
    {"item": "mlflow", "value": safe(lambda: __import__("mlflow").__version__)},
    {"item": "pandas", "value": pd.__version__},
    {"item": "scikit-learn", "value": safe(lambda: __import__("sklearn").__version__)},
    {"item": "xgboost", "value": safe(lambda: __import__("xgboost").__version__)},
])
display(versions)


In [ ]:


import os

try:
    spark.sparkContext
    spark_context = "AVAILABLE (classic cluster)"
except Exception:
    spark_context = "BLOCKED (serverless / restricted)"

probes = pd.DataFrame([
    {"probe": "spark.sparkContext", "result": spark_context},
    {"probe": "/dbfs FUSE mount", "result": "AVAILABLE" if os.path.exists("/dbfs") else "NOT MOUNTED"},
    {"probe": "spark.sql.ansi.enabled", "result": conf("spark.sql.ansi.enabled")},
    {"probe": "current_catalog()", "result": safe(lambda: spark.sql("SELECT current_catalog()").collect()[0][0])},
    {"probe": "current_schema()", "result": safe(lambda: spark.sql("SELECT current_schema()").collect()[0][0])},
    {"probe": "current_user()", "result": safe(lambda: spark.sql("SELECT current_user()").collect()[0][0])},
])
display(probes)


In [ ]:


def table_name(simple_name):
    return f"{SCHEMA}.{simple_name}"


resolution = pd.DataFrame([
    {"item": "CATALOG (tables)", "value": "none — 2-part schema.table on the old workspace"},
    {"item": "SCHEMA widget", "value": SCHEMA},
    {"item": "CATALOG (model registry / volume)", "value": "data_catalog() — Free Edition requires a catalog for models and volumes"},
    {"item": "example table name", "value": table_name("practice_customers")},
])
display(resolution)


In [ ]:


print("=== HOW TO READ THIS ===")
print("- OLD workspace (Databricks Free Edition): serverless, Python 3.10+, Spark 3.5.x, UC.")
print("- NEW workspace (Azure Databricks Premium + UC): serverless or a 15.x/16.x job cluster.")
print("- sparkContext BLOCKED on serverless: use the DataFrame API (this repo already does).")
print("- /dbfs FUSE missing on serverless: use /Volumes/... paths (this repo already does).")
print("- ANSI enabled on newer runtimes: avoid CAST on dirty strings; clean first or use try_cast.")
print(f"- resolved table naming for this session: {table_name('practice_customers')}")
print("- BOTH sides are Unity Catalog (3-part names): only the CATALOG is new on the Azure")
print("  side (e.g. <free_catalog>.practice.* -> practice_ml_dev.practice.*).")
print("- Classic fallback workspaces (CE / DBR 12.2) show 2-part names + available sparkContext.")
